# Experimento — TabPFN v2

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

Transformer pré-treinado da PriorLabs, usado aqui na versão 2, como classificador em contexto: o ajuste vê os clientes de treino da dobra e prevê a validação sem atualizar os pesos do checkpoint. Versões posteriores (v2.5 em diante) ficam fora deste notebook.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

O uso em contexto é `TabPFNClassifier.create_default_for_version` da versão 2. O construtor sem versão, no pacote atual, não carrega a v2. Os pesos ficam congelados. Camadas, largura, logística, tangente hiperbólica, ReLU, dropout e a escolha entre SGD e Adam não são argumentos desse construtor. Esta tabela cabe no teto da v2: 10.000 linhas e 500 colunas.

Cada item é um bloco. O default é o da v2 no código atual.

- `categorical_features_indices`. Default `None`: a v2 infere o tipo. Declarar o índice muda o tratamento da coluna. Numérica declarada como categórica só permanece categórica até 30 valores distintos.
- `balance_probabilities`. Default `False`. Repondera a probabilidade pela frequência das classes no treino. Não entra no mesmo bloco que o oversampling das dobras.
- `softmax_temperature`. Default 0,9. Valor menor deixa a previsão mais confiante, e 1,0 não altera o logit. Não há faixa oficial; a documentação exemplifica 0,8 e 1,2.
- `average_before_softmax`. Default `False`. Só age com mais de um estimador. Ligado, tira a média dos logits e só então aplica o softmax.
- `n_estimators`. Default `"auto"`, que nesta tabela resolve para 8. Um inteiro explícito não é aumentado depois. O teto do aumento automático é 32.
- `OUTLIER_REMOVAL_STD`. Default `"auto"`, que resolve para 12. `None` desliga. Não apaga linha: corta o extremo.
- `POLYNOMIAL_FEATURES`. Default `"no"`. `"all"` gera todos os produtos de dois fatores. Um inteiro limita quantos produtos entram.
- `PREPROCESS_TRANSFORMS`. A v2 de classificação reparte os estimadores em duas receitas: `quantile_uni_coarse`, com categórica ordinal embaralhada e SVD, e `none`, tratando categórica como número. Um bloco troca uma receita, contra esse par.
- `FINGERPRINT_FEATURE`. Default `True`. Acrescenta um hash da linha, que conta duplicata.
- `FEATURE_SHIFT_METHOD`. Default `"shuffle"`. As outras escolhas documentadas são `"rotate"` e `None`.
- `CLASS_SHIFT_METHOD`. Default `"shuffle"`. As outras escolhas documentadas são `"rotate"` e `None`.
- `tune_decision_thresholds`. Default desligado. Pede `eval_metric` em acurácia, F1 ou balanced accuracy. Com ROC-AUC ou log loss a chamada quebra.
- `calibrate_temperature`. Default desligado. Substitui a temperatura escrita à mão e não entra no mesmo bloco que `softmax_temperature`.

Nesta tabela não mudam a métrica, e ficam fora da busca: subamostra de colunas (o orçamento da v2 é 1.000.000 e há cerca de 20 colunas), expansão de texto e expansão de data. `device`, precisão e modo de memória também ficam fora.

Fine-tuning não existe dentro de `TabPFNClassifier`. Existe em `FinetunedTabPFNClassifier`, e na linha atual do pacote só mira a v2 se `model_version` for a v2. O otimizador é AdamW, sem seletor. Defaults da assinatura: `epochs` 30, `learning_rate` 1e-5, `weight_decay` 0,01, early stopping ligado, `early_stopping_patience` 8, `n_estimators_finetune` 2 e `n_estimators_final_inference` 8. A documentação sugere, como ponto de partida e não como grade, 10 a 30 épocas e taxa de 1e-5 até 1e-4 numa tabela deste tamanho. O bloco de fine-tuning só abre depois dos blocos em contexto, e só fica se vencer o checkpoint congelado. Dentro dele, taxa, épocas, paciência e `weight_decay` são blocos separados. Não há dropout nem troca de ativação.

O oversampling repetitivo das dobras vale no bloco 0.

Fontes: a documentação PriorLabs do classificador, da `InferenceConfig` e do fine-tuning, e `src/tabpfn/classifier.py` no repositório PriorLabs/TabPFN.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. TabPFN v2 em contexto, com `create_default_for_version` da v2, sobre as dobras com oversampling.
2. `categorical_features_indices`.
3. `balance_probabilities`, sem mexer no oversampling.
4. `softmax_temperature`, sem calibração automática no mesmo bloco.
5. `average_before_softmax`.
6. `n_estimators`.
7. `OUTLIER_REMOVAL_STD`.
8. `POLYNOMIAL_FEATURES`.
9. `PREPROCESS_TRANSFORMS`, uma receita de cada vez, contra o par default da v2.
10. `FINGERPRINT_FEATURE`.
11. `FEATURE_SHIFT_METHOD`.
12. `CLASS_SHIFT_METHOD`.
13. `tune_decision_thresholds`, com a métrica de decisão em acurácia, F1 ou balanced accuracy.
14. `calibrate_temperature`, no lugar do bloco da temperatura manual.
15. Subamostra com `majority_downsample`, junto com um `SUBSAMPLE_SAMPLES` acima da contagem da classe minoritária. O método sozinho não age.
16. Fine-tuning da v2, se for aberto. Primeiro o portão, com os defaults. Se vencer: taxa, épocas, paciência e `weight_decay`, um por bloco.
17. Estratégias de dados compartilhadas.
18. Bloco final no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
